# Galaxy Classifier — Colab training

Trains the EfficientNet-B0 classifier from `ml/train.py` on a Colab T4 GPU.

In [3]:
!nvidia-smi | head -20

'head' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# EDIT THIS if your project lives under a different path on Drive.
DRIVE_PROJECT = '/content/drive/MyDrive/galaxy-classification'
LOCAL_PROJECT = '/content/galaxy-classification'

import os
assert os.path.isdir(DRIVE_PROJECT), f'Not found on Drive: {DRIVE_PROJECT}'
for must_exist in ('ml/train.py', 'data/processed/train.csv',
                   'data/processed/val.csv', 'data/processed/test.csv',
                   'data/raw/images_gz2.zip'):
    p = os.path.join(DRIVE_PROJECT, must_exist)
    assert os.path.exists(p), f'Missing on Drive: {p}'
print('Drive project ready.')

In [ ]:
# Copy project to local /content (fast disk, immune to Drive disconnects).
# Skips files that are already copied so re-runs are cheap.
import os, shutil, time

os.makedirs(f'{LOCAL_PROJECT}/ml', exist_ok=True)
os.makedirs(f'{LOCAL_PROJECT}/data/processed', exist_ok=True)
os.makedirs(f'{LOCAL_PROJECT}/data/raw', exist_ok=True)
os.makedirs(f'{LOCAL_PROJECT}/ml/artifacts', exist_ok=True)

def copy_if_needed(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f'  skip (already copied): {dst}')
        return
    t0 = time.time()
    shutil.copy(src, dst)
    mb = os.path.getsize(dst) / 1e6
    print(f'  copied {mb:.1f} MB in {time.time()-t0:.1f}s -> {dst}')

for fname in os.listdir(f'{DRIVE_PROJECT}/ml'):
    if fname.endswith('.py'):
        copy_if_needed(f'{DRIVE_PROJECT}/ml/{fname}', f'{LOCAL_PROJECT}/ml/{fname}')
for fname in ('train.csv', 'val.csv', 'test.csv'):
    copy_if_needed(f'{DRIVE_PROJECT}/data/processed/{fname}', f'{LOCAL_PROJECT}/data/processed/{fname}')
copy_if_needed(f'{DRIVE_PROJECT}/data/raw/images_gz2.zip', f'{LOCAL_PROJECT}/data/raw/images_gz2.zip')
print('Local copy ready at', LOCAL_PROJECT)

In [ ]:
!pip install -q onnxscript

In [ ]:
# Train. CWD stays at /content (NOT the Drive mount).
%cd /content
!python {LOCAL_PROJECT}/ml/train.py \
  --device cuda \
  --amp \
  --epochs 8 \
  --batch-size 128 \
  --workers 4 \
  --img-size 224 \
  --lr 3e-4

In [ ]:
# Quick test-set evaluation using the freshly trained best.pt.
import sys, torch
from torch.utils.data import DataLoader

sys.path.insert(0, f'{LOCAL_PROJECT}/ml')
from class_map import CLASSES
from dataset import GalaxyZipDataset, build_transforms
from train import build_model

device = torch.device('cuda')
model = build_model(num_classes=len(CLASSES), freeze_backbone=False).to(device)
ckpt = torch.load(f'{LOCAL_PROJECT}/ml/artifacts/best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()

tfm = build_transforms(ckpt['img_size'], train=False)
ds = GalaxyZipDataset(f'{LOCAL_PROJECT}/data/processed/test.csv',
                      f'{LOCAL_PROJECT}/data/raw/images_gz2.zip', tfm)
dl = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False)

correct = 0; total = 0
per_cls_correct = [0]*len(CLASSES); per_cls_total = [0]*len(CLASSES)
with torch.no_grad():
    for x, y in dl:
        x = x.to(device); y = y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += x.size(0)
        for c in range(len(CLASSES)):
            mask = (y == c)
            per_cls_total[c] += int(mask.sum())
            per_cls_correct[c] += int(((pred == y) & mask).sum())
print(f'Test accuracy: {correct/total:.4f} ({correct}/{total})')
for i, c in enumerate(CLASSES):
    if per_cls_total[i]:
        print(f'  {c:15s} {per_cls_correct[i]/per_cls_total[i]:.4f}  ({per_cls_correct[i]}/{per_cls_total[i]})')

In [ ]:
# Copy trained artifacts back to Drive so they sync to your PC.
import os, shutil
os.makedirs(f'{DRIVE_PROJECT}/ml/artifacts', exist_ok=True)
for fname in os.listdir(f'{LOCAL_PROJECT}/ml/artifacts'):
    src = f'{LOCAL_PROJECT}/ml/artifacts/{fname}'
    dst = f'{DRIVE_PROJECT}/ml/artifacts/{fname}'
    shutil.copy(src, dst)
    print(f'  -> {dst}  ({os.path.getsize(dst)/1e6:.2f} MB)')
print('Done. Artifacts are now in Drive and will sync to your PC if Drive Desktop is installed.')